# ShopDesk, Section 2 Lab 1: The Agentic Loop

A beginner-friendly notebook that builds the **agentic loop** for ShopDesk. We first
implement it by hand on the **base Anthropic SDK** so `stop_reason` is fully visible, then
show the same loop **productised** by the **Claude Agent SDK**'s `query()`. Both run
**Sonnet** (`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

In Section 1 an agent handled one tool call, one round-trip. But a real request ("check
A2's status, then refund it") needs several tool calls in a row, and you cannot know in
advance how many. A **loop** solves this: keep going while the model asks for tools, stop
when it says it is done. The signal that tells you which is `stop_reason`.

The question this lab answers: **how do you drive an agent by the model's own signals
instead of a hardcoded number of steps?**

## Objectives

- Implement the loop lifecycle where iteration continues on `stop_reason`, not on fixed
  logic.
- Handle `stop_reason` explicitly: `tool_use` continues, `end_turn` cleanly ends.
- Prefer **model-driven** control flow over a **rule-based** hardcoded workflow, and see
  why arbitrary loop limits and ignoring `stop_reason` are anti-patterns.
- See the Claude Agent SDK's `query()` as this same loop, already built for you.

## What you'll observe

- On a two-step request, `stop_reason` prints `tool_use` while work remains, then
  `end_turn` once at the end.
- The rule-based loop does the same fixed steps every time, even when the request did not
  ask for them; the model-driven loop does only what the request needs.
- The Agent SDK's `query()` drives the identical loop with no hand-written control flow.

## How to run

Run top to bottom. The rule-based loop and all the building blocks are pure Python and run
anywhere. The model-driven cells call Claude, so paste a real key into **Setup 2/3** and
re-run from the top; otherwise they skip cleanly. **Node.js 18+** must be installed for the
Agent SDK section.

## 0. Setup

**This cell:** installs the packages. We install the **Agent SDK** (for the
productised loop) and the **base Anthropic SDK** (for the hand-written loop where
`stop_reason` is visible). The Agent SDK also needs Node.js 18+, which cannot be
pip-installed.

In [ ]:
# ===== SETUP 1/3 - install both SDKs =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports what we need, pins the model, and sets a `RUN_LIVE` switch so
live calls fire only with a real key. It also imports the async and threading helpers the
Agent SDK section uses later.

In [ ]:
# ===== SETUP 2/3 - imports, the model, and a live/offline switch =====
import os                                       # read the API key from the environment
import sys                                      # detect Windows (it needs a special event loop)
import json                                     # print tool payloads readably
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)
import anthropic                                # the base Anthropic SDK (synchronous)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds the shared **ShopDesk world** and its tools. Two orders, plus
the status and refund tools the loop will call. This is the environment every loop below
acts on.

In [ ]:
# ===== SETUP 3/3 - the shared data and the two tools =====
ORDERS = {                                       # our tiny order book
    "A1": {"status": 2, "refundable": True},     #   shipped,   within the window
    "A2": {"status": 3, "refundable": False},    #   delivered, past the window
}
STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}   # status code -> human word

def get_order_status(order_id):                  # tool 1: look up a status
    o = ORDERS.get(order_id)                     #   find the order
    return STATUS_NAMES[o["status"]] if o else "unknown order"   # status word or a miss

def refund_order(order_id):                      # tool 2: attempt a refund
    o = ORDERS.get(order_id)                     #   find the order
    if not o:              return "unknown order"          #   no such order
    if not o["refundable"]: return "refused: past 30-day window"   #   the business rule
    return "refunded"                            #   success

RUN_TOOL = {"get_order_status": get_order_status, "refund_order": refund_order}   # name -> function
print("tools:", list(RUN_TOOL))                  # confirm both exist

**This cell:** the **schemas** the model reads to choose a tool. Same two tools, each
taking one `order_id`. The loop hands this list to the model on every iteration.

In [ ]:
# ===== the tool schemas =====
_arg = {"type": "object",                         # both tools take one string order_id
        "properties": {"order_id": {"type": "string"}},
        "required": ["order_id"]}
TOOLS = [                                          # the tool list the model sees each turn
    {"name": "get_order_status",
     "description": "Look up the delivery status of an order.", "input_schema": _arg},
    {"name": "refund_order",
     "description": "Refund an order if it is within the 30-day window.", "input_schema": _arg},
]
print("schemas ready:", [t["name"] for t in TOOLS])   # confirm

---

### 🎯 Lab objective - drive the agent by `stop_reason`

**What you build:** the agentic loop by hand, then the same loop through the Agent SDK,
plus a rule-based version to compare against.

**Why it helps you build real solutions:** the loop is the engine under every agent
framework. Once you read `stop_reason` correctly, you never guess when an agent is done and
never cap it at an arbitrary number of steps.

**How you'll see it:** `stop_reason` stays `tool_use` until the work is finished, then flips
to `end_turn`, and the model decides how many steps that takes.

**This cell:** the **agentic loop**, written by hand. The rule is simple: on
`tool_use`, run the requested tools, **append** their results, and loop; on `end_turn`,
stop. The `range()` is only a safety net, not the exit condition; `stop_reason` is what
actually ends it.

In [ ]:
# ===== the model-driven loop, driven only by stop_reason =====
def run_loop(question, max_guard=8):              # run ShopDesk until the model says it is done
    client = anthropic.Anthropic()                #   the LLM client (reads the key)
    messages = [{"role": "user", "content": question}]   # the running conversation
    for step in range(max_guard):                 #   a generous SAFETY NET, not the exit rule
        r = client.messages.create(model=MODEL, max_tokens=512,   # one model turn
                                   tools=TOOLS, messages=messages)
        print(f"step {step}: stop_reason =", r.stop_reason)   #   THE signal we branch on
        if r.stop_reason == "end_turn":           #   model says it is finished
            return "".join(b.text for b in r.content if b.type == "text")   # the final answer
        messages.append({"role": "assistant", "content": r.content})   # keep the model's turn
        results = []                              #   collect this turn's tool results
        for b in r.content:                       #   walk the blocks the model produced
            if b.type == "tool_use":              #     it asked for a tool
                out = RUN_TOOL[b.name](**b.input) #       run the real Python function
                print("  ran", b.name, b.input, "->", out)   #     show what happened
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": out})
        messages.append({"role": "user", "content": results})   # APPEND results -> model sees them next loop
    return "(stopped: hit the safety guard)"      #   only reached if the guard trips

**This cell:** runs the loop on a request that needs two tools **in order**: check
A2's status, then refund it. Watch `stop_reason` stay `tool_use` across the steps, then flip
to `end_turn` once the work is genuinely done.

In [ ]:
# ===== run the model-driven loop on a two-step request =====
if RUN_LIVE:                                      # needs a real key
    print("ANSWER:", run_loop("Check order A2's status, then refund it."))
else:
    print("[skipped - set ANTHROPIC_API_KEY to run this live]")

**This cell:** a **rule-based** loop for contrast: a hardcoded workflow that always
checks status and then always refunds, in that fixed order, no matter the request. It is
pure Python, so it runs offline; note that nothing about it can adapt.

In [ ]:
# ===== the rule-based (hardcoded) alternative =====
def run_rulebased(order_id):                      # a FIXED workflow: no model decides control flow
    status = get_order_status(order_id)           #   step 1: always check status
    refund = refund_order(order_id)               #   step 2: always attempt a refund
    return f"status={status}; refund={refund}"    #   the same two steps, every time

**This cell:** compares the two on a **shipping-only** request. The rule-based version
refunds A1 anyway (the customer never asked), while the model-driven loop calls only the
status tool and stops. This is the cost of hardcoding control flow: it cannot match the
work to the request.

In [ ]:
# ===== compare: shipping-only request, "Where is A1?" =====
print("rule-based (always refunds too):", run_rulebased("A1"))   # runs offline; note the unwanted refund
if RUN_LIVE:                                      # the model-driven loop, for contrast
    print("model-driven (does only what is asked):", run_loop("Where is order A1 right now?"))
else:
    print("[model-driven skipped] expected: it checks status, then end_turn, with NO refund.")

**This cell:** defines `run_async()`, a notebook-safe wrapper that runs any async
Agent SDK call in its own thread and event loop. We build it once so the Agent SDK cells can
be called like ordinary functions.

In [ ]:
# ===== a notebook-safe runner for async Agent SDK calls =====
def run_async(make_coro):                         # make_coro: a function returning a coroutine
    box = {}                                      #   carries the result or error out of the thread
    def worker():                                 #   runs in its own thread
        if sys.platform == "win32":               #     Windows needs the Proactor loop...
            loop = asyncio.ProactorEventLoop()    #       ...to spawn the Agent SDK subprocess
        else:                                     #     macOS / Linux:
            loop = asyncio.new_event_loop()       #       a plain new loop is fine
        asyncio.set_event_loop(loop)              #     make it this thread's loop
        try:                                      #
            box["value"] = loop.run_until_complete(make_coro())   # run the coroutine to completion
        except Exception as e:                    #     capture any error...
            box["error"] = e                      #       ...to re-raise on the main thread
        finally:                                  #
            loop.close()                          #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box:                            #   worker failed?
        raise box["error"]                        #     surface the error here
    return box.get("value")                       #   hand back the result

**This cell:** imports the Agent SDK pieces and defines `stream_run()`, which runs one
`query()` and prints each tool call plus the final result. `query()` **is** the loop from
Part A, already built: it iterates internally until the model stops.

In [ ]:
# ===== stream one Agent SDK query() and narrate it =====
from claude_agent_sdk import (                     # the Agent SDK pieces we use:
    query, ClaudeAgentOptions,                     #   run + options
    tool, create_sdk_mcp_server,                   #   define a tool + bundle it
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,   # message + block types
)

async def stream_run(options, prompt):             # run query() and print what happens
    print("USER:", prompt)                         #   echo the request
    answer = ""                                    #   keep the final text
    async for message in query(prompt=prompt, options=options):   # stream every message
        if isinstance(message, AssistantMessage):  #     the model took a turn
            for block in message.content:          #       look at each block
                if isinstance(block, ToolUseBlock):#       it called a tool...
                    print("  -> tool:", block.name.split("__")[-1], block.input)
                elif isinstance(block, TextBlock):  #      ...or produced text
                    answer = block.text            #         remember the latest text
        elif isinstance(message, ResultMessage):    #     the run finished
            print("  (query finished; session:", getattr(message, "session_id", None), ")")
    print("ANSWER:", answer)                        #   the final answer
    return answer

**This cell:** gives the Agent SDK a small tool and options so `query()` has something
to loop over. `create_sdk_mcp_server` bundles the tool; `allowed_tools` lets the model call
it. This is the productised equivalent of the schema plus `RUN_TOOL` dispatch we wrote by
hand.

In [ ]:
# ===== a tiny Agent SDK tool + options =====
@tool("get_order_status", "Look up the delivery status of an order.", {"order_id": str})
async def sdk_status(args):                        # the same status lookup, as an SDK tool
    oid = args["order_id"]                         #   which order
    o = ORDERS.get(oid)                            #   find it
    word = STATUS_NAMES[o["status"]] if o else "unknown order"   # status word or a miss
    return {"content": [{"type": "text", "text": word}]}   # MCP result shape

shop = create_sdk_mcp_server(name="shop", version="1.0.0", tools=[sdk_status])   # bundle the tool
OPTS = ClaudeAgentOptions(                          # options for the productised loop
    model=MODEL, mcp_servers={"shop": shop},        #   the model + our tool server
    allowed_tools=["mcp__shop__get_order_status"])  #   permit exactly this tool

**This cell:** runs `query()` on a status request. It drives the same loop as Part A:
the model calls the tool, consumes the result, and answers, all without a hand-written
control flow. This is what you would reach for in practice.

In [ ]:
# ===== run the productised loop =====
if RUN_LIVE:                                      # needs a real key (and Node.js 18+)
    run_async(lambda: stream_run(OPTS, "What is the status of order A1?"))
else:
    print("[skipped - set ANTHROPIC_API_KEY to run this live]")

| anti-pattern | what to do instead |
|---|---|
| cap the loop at `range(3)` and hope | let `end_turn` end it; keep a range only as a safety net |
| scan the reply for words like "done" | branch on `stop_reason`, never on prose |
| ignore `stop_reason` and always re-ask | append every `tool_result`, then loop |
| hardcode the sequence of tool calls | let the model choose the steps the request needs |

**Lesson:** the model decides *how many* steps ShopDesk needs; your code decides *when
to stop* by reading `stop_reason`. A rule-based workflow cannot adapt; a model-driven loop
does exactly the work the request calls for. The Agent SDK's `query()` is this loop already
built, which is what the next lab uses when it manages sessions.

---

## Recap - the loop, three ways

| Approach | Control flow | When to use |
|---|---|---|
| Model-driven loop (base SDK) | continues on `tool_use`, stops on `end_turn` | to see and own the mechanism |
| Rule-based workflow | fixed steps, no decisions | only when steps truly never vary |
| Agent SDK `query()` | the same loop, productised | in practice, once you trust the loop |

One principle to carry forward: **the model reasons and signals with `stop_reason`; your
code loops on that signal, never on a hardcoded count.** To run live, paste a real key into
**Setup 2/3** and re-run from the top. Then try it: give the loop a three-step request and
watch it take exactly three tool turns before `end_turn`. Next lab: keeping this
conversation across sessions with resume and fork.